# December 2025 Solution

I modeled both players with threshold strategies.

- I let `u` be my reroll threshold.
- I let `v` be the other player's reroll threshold.
- I let `s` be the cutoff used for the information leak.

The whole solution becomes a sequence of three symbolic calculations:

1. find the symmetric equilibrium of the fair game,
2. let the second player exploit the leaked information,
3. then let the first player counter that exploit.

I used `sympy` for the algebra so I could keep everything exact.


In [1]:
from sympy import Rational, diff, simplify, solve, sqrt
from sympy.abc import u, v, s


## Fair Game

I started with the baseline game where both players only know their own first throw.

If I reroll below `u` and the other player rerolls below `v`, then there are four natural cases depending on whether each first throw is kept or rerolled. I wrote my win probability as the sum of those four contributions.


In [9]:
def fair_game_win_probability():
    both_reroll = u * v * Rational(1, 2)
    only_i_reroll = u * (1 - v) * ((1 - v) / 2)
    only_other_rerolls = (1 - u) * v * ((1 + u) / 2)
    neither_rerolls = (1 - u) * (1 - v) * ((1 - v) / (2 * (1 - u)))
    return simplify(both_reroll + only_i_reroll + only_other_rerolls + neither_rerolls)


p_fair = fair_game_win_probability()
p_fair


-u**2*v/2 + u*v**2/2 - u*v/2 + u/2 + v**2/2 - v/2 + 1/2

To get the fair-game threshold, I first found the other player's best response as a function of `u`, and then I chose the value of `u` that maximizes my resulting win probability.


In [16]:
def fair_equilibrium_threshold():
    best_reply = simplify(solve(diff(p_fair, v), v)[0])
    reduced_game = simplify(p_fair.subs(v, best_reply))
    candidates = solve(diff(reduced_game, u), u)
    return simplify(candidates[2])


u_star = fair_equilibrium_threshold()
u_star


-1/2 + sqrt(5)/2

So the fair-game threshold is

`u* = (sqrt(5) - 1) / 2`.


## Using the Leak

Next I assumed the other player learns whether my first throw is below the equilibrium threshold `u*`.

If my first throw is below that cutoff, they know I will reroll, so their best move in that branch is different from the branch where they know I keep my first number. I encoded that split directly into a new win-probability formula.


In [22]:
def leaked_game_win_probability():
    low_and_other_rerolls = (u / 2) * Rational(1, 2)
    low_and_other_keeps = (u / 2) * Rational(1, 4)
    high_and_other_rerolls = (1 - u) * v * ((1 + u) / 2)
    high_and_other_keeps = (1 - u) * (1 - v) * ((1 - v) / (2 * (1 - u)))
    return simplify(
        low_and_other_rerolls
        + low_and_other_keeps
        + high_and_other_rerolls
        + high_and_other_keeps
    )


p_leaked = leaked_game_win_probability()
p_leaked


3*u/8 - v*(u - 1)*(u + 1)/2 + (v - 1)**2/2

Then I fixed my threshold at the fair-game value and let the other player choose the threshold that minimizes my win probability in the branch where my first throw is not leaked as a reroll.


In [27]:
def leaked_game_best_reply():
    candidate = solve(diff(p_leaked.subs(u, u_star), v), v)[0]
    return simplify(candidate)


v_star = leaked_game_best_reply()
v_star


5/4 - sqrt(5)/4

That gives the other player's best reply

`v* = (5 - sqrt(5)) / 4`.


## Counterplay

At that point I stopped assuming I had to keep using the fair-game threshold.

If the other player is committing to the leak-based strategy, then I can answer with a new threshold `u` below the leaked cutoff `s`. That creates six distinct branches:

- I reroll and they reroll.
- I reroll and they keep.
- I keep a value between `u` and `s` while they reroll.
- I keep a value between `u` and `s` while they keep.
- I keep a value above `s` while they reroll.
- I keep a value above `s` while they keep.

I added those six contributions directly.


In [28]:
def counterplay_win_probability():
    case_1 = (u / 2) * Rational(1, 2)
    case_2 = (u / 2) * Rational(1, 4)
    case_3 = ((s - u) / 2) * ((s + u) / 2)
    case_4 = ((s - u) / 2) * (s + u - 1)
    case_5 = (1 - s) * v * ((1 + s) / 2)
    case_6 = (1 - s) * (1 - v) * ((1 - v) / (2 * (1 - s)))
    return simplify(case_1 + case_2 + case_3 + case_4 + case_5 + case_6)


p_counter = counterplay_win_probability()
p_counter


-s**2*v/2 + 3*s**2/4 - s/2 - 3*u**2/4 + 7*u/8 + v**2/2 - v/2 + 1/2

Now I substituted the leaked cutoff `s = u*` and the opponent's best response `v = v*`, and then optimized over my new threshold `u`.


In [29]:
def optimal_counterplay():
    target = simplify(p_counter.subs({s: u_star, v: v_star}))
    best_u = solve(diff(target, u), u)[0]
    best_p = simplify(target.subs(u, best_u))
    return simplify(best_u), simplify(best_p)


u_counter, p_counter_star = optimal_counterplay()
u_counter, p_counter_star


(7/12, 229/192 - 5*sqrt(5)/16)

So the counter-strategy threshold is

`u = 7/12`,

and the resulting win probability is

`(229 - 60*sqrt(5)) / 192`.


In [30]:
def show_results():
    values = {
        'fair threshold': u_star,
        'opponent best reply': v_star,
        'my counter threshold': u_counter,
        'final win probability': p_counter_star,
    }
    for label, expr in values.items():
        print(f"{label}: {expr} = {expr.evalf()}")


show_results()


fair threshold: -1/2 + sqrt(5)/2 = 0.618033988749895
opponent best reply: 5/4 - sqrt(5)/4 = 0.690983005625053
my counter threshold: 7/12 = 0.583333333333333
final win probability: 229/192 - 5*sqrt(5)/16 = 0.493937090364649
